# 07 - Qiskit V2 Primitives (BaseSamplerV2 / BaseEstimatorV2)

Qiskit removed the V1 primitives (`qiskit.primitives.Sampler`,
`qiskit.primitives.Estimator`, and the bare `BaseSampler`/`BaseEstimator`
names) in `qiskit>=2.0`. `wrap_qiskit_sampler` and `wrap_qiskit_estimator`
now require **V2 primitives** (`BaseSamplerV2` / `BaseEstimatorV2`) --
`StatevectorSampler`, `StatevectorEstimator`, `BackendSamplerV2`,
`BackendEstimatorV2`, or a real-hardware V2 primitive all work.

This notebook covers:

1. The `StatevectorSampler` classification path (same shape of usage as the
   other quickstarts, just on the current primitive API).
2. The `StatevectorEstimator` path, for both classification (multiple
   observables -> softmax) and regression (single observable).
3. **One gotcha that will silently break shot-based uncertainty methods**:
   how you seed a V2 primitive controls whether repeated calls actually
   produce different shot noise.


In [1]:
import numpy as np
from qiskit.circuit import Parameter, QuantumCircuit
from qiskit.primitives import StatevectorEstimator, StatevectorSampler
from qiskit.quantum_info import SparsePauliOp

from quantumuq import (
    ShotBootstrap,
    ece,
    nll,
    wrap_qiskit_estimator,
    wrap_qiskit_sampler,
)
from quantumuq.datasets.toy import make_moons

rng = np.random.default_rng(0)
dataset = make_moons(n_samples=200, noise=0.1, random_state=0)
X, y = dataset.X, dataset.y
perm = rng.permutation(len(X))
train_idx, test_idx = perm[:150], perm[150:]
X_test, y_test = X[test_idx], y[test_idx]


def feature_map(X_arr: np.ndarray):
    X_arr = np.atleast_2d(X_arr)
    return [[float(x[0])] for x in X_arr]

## 1. The seeding gotcha, up front

A plain integer `seed` re-seeds a fresh RNG on *every* `.run()` call, so a
`StatevectorSampler(seed=0)` returns bit-for-bit identical counts each time
it's called with the same input. That silently defeats `ShotBootstrap`,
`NoiseProfile`, and anything else that relies on repeated calls seeing
independent shot noise -- the reported uncertainty would come out as exactly
zero, which looks like a bug in the library rather than in how the primitive
was constructed.

Passing a `numpy.random.Generator` **instance** instead makes the RNG state
advance across calls, so repeated draws differ (while the whole notebook
stays reproducible, since the Generator itself is seeded).

In [2]:
theta = Parameter("theta")
qc_meas = QuantumCircuit(1)
qc_meas.ry(theta, 0)
qc_meas.measure_all()

bad_sampler = StatevectorSampler(seed=0)  # plain int -- resets every call
good_sampler = StatevectorSampler(seed=np.random.default_rng(0))  # Generator -- advances

for label, sampler in [("seed=0 (int)", bad_sampler), ("seed=Generator", good_sampler)]:
    predictor = wrap_qiskit_sampler(
        sampler, circuit=qc_meas, task="classification", n_classes=2, feature_map=feature_map
    )
    model = predictor.with_uq(ShotBootstrap(n_samples=8, shots=500, seed=0))
    dist = model.predict_dist(np.array([[1.0]]))
    print(f"{label:16s} -> ShotBootstrap std = {dist.std.max():.5f}")

seed=0 (int)     -> ShotBootstrap std = 0.00000
seed=Generator   -> ShotBootstrap std = 0.01884


## 2. Sampler classification, end to end

In [3]:
sampler = StatevectorSampler(seed=np.random.default_rng(0))
sampler_predictor = wrap_qiskit_sampler(
    sampler, circuit=qc_meas, task="classification", n_classes=2, feature_map=feature_map
)
uq = ShotBootstrap(n_samples=8, shots=1000, seed=0)
sampler_model = sampler_predictor.with_uq(uq)

dist = sampler_model.predict_dist(X_test)
probs = dist.mean
print(f"Test NLL: {nll(y_test, probs):.3f}, ECE: {ece(y_test, probs, n_bins=10):.3f}")
print("dist.mean.shape:", dist.mean.shape, " dist.std.shape:", dist.std.shape)

Test NLL: 1.071, ECE: 0.271
dist.mean.shape: (50, 2)  dist.std.shape: (50, 2)


## 3. Estimator: classification and regression

Estimator circuits must **not** contain measurement instructions -- the
estimator computes expectation values directly rather than sampling
bitstrings, so `circuit.measure_all()` is a Sampler-only concern.

V2 estimators also don't take a `shots` argument directly; they target a
`precision` (standard error). For a Pauli observable (eigenvalues +/-1), the
standard error over `shots` samples is at most `1/sqrt(shots)`, which is the
conversion `wrap_qiskit_estimator` applies internally when you pass `shots=`.

In [4]:
qc_obs = QuantumCircuit(1)
qc_obs.ry(theta, 0)

estimator = StatevectorEstimator(seed=np.random.default_rng(0))

# Classification: one observable per class, softmax by default.
est_clf_predictor = wrap_qiskit_estimator(
    estimator,
    circuit=qc_obs,
    observables=[SparsePauliOp("Z"), SparsePauliOp("X")],
    task="classification",
    n_classes=2,
    feature_map=feature_map,
)
clf_probs = est_clf_predictor.predict_proba(X_test, shots=1000)
print("Estimator classification probs shape:", clf_probs.shape, " rows sum to 1:", np.allclose(clf_probs.sum(axis=1), 1.0))

# Regression: single observable, expectation value returned directly.
est_reg_predictor = wrap_qiskit_estimator(
    estimator,
    circuit=qc_obs,
    observables=[SparsePauliOp("Z")],
    task="regression",
    feature_map=feature_map,
)
y_reg = est_reg_predictor.predict(np.array([[0.0], [np.pi]]), shots=1000)
print("<Z> at theta=0 (expect ~+1):", y_reg[0, 0])
print("<Z> at theta=pi (expect ~-1):", y_reg[1, 0])

Estimator classification probs shape: (50, 2)  rows sum to 1: True
<Z> at theta=0 (expect ~+1): 1.0158962274630907
<Z> at theta=pi (expect ~-1): -0.9687025258486281


## Summary

- `wrap_qiskit_sampler`/`wrap_qiskit_estimator` require V2 primitives
  (`BaseSamplerV2`/`BaseEstimatorV2`) -- pass `StatevectorSampler`,
  `StatevectorEstimator`, `BackendSamplerV2`, `BackendEstimatorV2`, or a
  real-backend V2 primitive.
- Seed V2 primitives with a `numpy.random.Generator` instance, not a plain
  int, if you want `ShotBootstrap`/`NoiseProfile` to see real shot noise
  across repeated calls.
- Estimator circuits must not contain measurements; Sampler circuits must.
- `shots=` on the estimator path is converted to `precision = 1/sqrt(shots)`.
- Tests: `tests/test_qiskit_adapter.py`.